# 🛠️ GoldMind - Stationary Feature Engineering
This notebook generates 45+ scale-invariant, stationary technical indicators for XAU/USD.

### ⚠️ Critical Note on Stationarity in Machine Learning
Tree models (Random Forest, XGBoost) split on raw numeric thresholds. 
If raw price levels (e.g. `ma_200`, `close_lag_1h`, `bb_upper`) are used as features:
- In Gold's bull market (e.g. \$2,000 in train $\rightarrow$ \$5,000 in test), **every test sample falls outside the training range**.
- The tree maps all test data to a single extreme leaf node, causing catastrophic bias.
- **Solution:** All indicators MUST be normalized relative to current price or bounded (e.g. `Close/MA - 1`, `%B`, normalized ATR, return lags, cyclical time encodings).


In [ ]:
import os
import sys

# Ensure repository root is on Python path
sys.path.insert(0, os.path.abspath("."))

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt

# Import from our modular package
from src.features import build_features, make_target, select_top_features

CSV_PATH = "data1/XAU_1m_dataX.csv"
OUT_DIR = "model_output"
os.makedirs(OUT_DIR, exist_ok=True)
print("Features setup ready.")


Features setup ready.


In [2]:
# ---------- 1. Load 1-min Data and Resample to 1-Hour ----------
print(f"Loading {CSV_PATH} ...")
raw = pd.read_csv(CSV_PATH, parse_dates=["Date"]).set_index("Date").sort_index()

df_1h = (
    raw.resample("1h")
    .agg({"Open": "first", "High": "max", "Low": "min", "Close": "last", "Volume": "sum"})
    .dropna()
)
print(f"Hourly dataset: {len(df_1h):,} bars ({df_1h.index.min()} to {df_1h.index.max()})")


Loading data/XAU_1m_data.csv ...
Hourly dataset: 12,230 bars (2024-01-02 01:00:00 to 2026-02-27 05:00:00)


In [3]:
# ---------- 2. Build Stationary Features ----------
print("Building stationary features via src.features.build_features() ...")
feats = build_features(df_1h)
y_reg = make_target(df_1h, horizon=1, kind="regression")
y_clf = make_target(df_1h, horizon=1, kind="classification")

print(f"Generated {feats.shape[1]} features across {len(feats):,} bars.")
print("\nFeature categories created:")
print("- Multi-horizon returns: ret_1 to ret_34, ret_240")
print("- Autoregressive past 1-bar returns: ret_lag_1h to ret_lag_10h (shift-based)")
print("- Relative MA distance: px_over_ma_5 to px_over_ma_200 (Close / MA - 1)")
print("- Volatility & Normalized ATR: vol_5 to vol_50, atr_pct_14, atr_pct_50")
print("- Normalized Bollinger Bands: bb_pct_b, bb_width")
print("- Wilder's RSI: rsi_7, rsi_14, rsi_21")
print("- Normalized MACD: macd_norm, macd_signal_norm, macd_hist_norm")
print("- Volume z-score & ratios: vol_zscore_20, vol_ratio_ma_5, etc.")
print("- Candle geometry: candle_body, candle_upper_wick, candle_lower_wick")
print("- Session & Cyclical Time: hour_sin, hour_cos, dow_sin, dow_cos, is_market_gap")


Building stationary features via src.features.build_features() ...
Generated 52 features across 12,230 bars.

Feature categories created:
- Multi-horizon returns: ret_1 to ret_34, ret_240
- Autoregressive past 1-bar returns: ret_lag_1h to ret_lag_10h (shift-based)
- Relative MA distance: px_over_ma_5 to px_over_ma_200 (Close / MA - 1)
- Volatility & Normalized ATR: vol_5 to vol_50, atr_pct_14, atr_pct_50
- Normalized Bollinger Bands: bb_pct_b, bb_width
- Wilder's RSI: rsi_7, rsi_14, rsi_21
- Normalized MACD: macd_norm, macd_signal_norm, macd_hist_norm
- Volume z-score & ratios: vol_zscore_20, vol_ratio_ma_5, etc.
- Candle geometry: candle_body, candle_upper_wick, candle_lower_wick
- Session & Cyclical Time: hour_sin, hour_cos, dow_sin, dow_cos, is_market_gap


In [4]:
# ---------- 3. Verify Stationarity & Clean Data ----------
# Combine features and regression target
data = feats.join(y_reg.rename("target")).dropna()
X = data.drop(columns=["target"])
y = data["target"]

print(f"Clean samples after dropping warmup rolling windows: {len(data):,}")

# Check that NO raw absolute price columns exist
raw_price_cols = [c for c in X.columns if c in ["Close", "High", "Low", "Open", "ma_20", "bb_upper", "bb_lower"]]
if len(raw_price_cols) == 0:
    print("✅ Stationarity check passed: No raw dollar price levels found in feature set.")
else:
    print("⚠️ WARNING: Found raw price columns:", raw_price_cols)

print("\nSample feature summary:")
print(X[["ret_1", "ret_lag_1h", "px_over_ma_20", "atr_pct_14", "rsi_14", "bb_pct_b", "hour_sin"]].describe().round(4))


Clean samples after dropping warmup rolling windows: 11,989
✅ Stationarity check passed: No raw dollar price levels found in feature set.

Sample feature summary:
            ret_1  ret_lag_1h  px_over_ma_20  atr_pct_14      rsi_14  \
count  11989.0000  11989.0000     11989.0000  11989.0000  11989.0000   
mean       0.0001      0.0001         0.0007      0.0032     53.1641   
std        0.0029      0.0029         0.0071      0.0020     12.7549   
min       -0.0573     -0.0573        -0.0728      0.0010     12.2698   
25%       -0.0009     -0.0009        -0.0020      0.0021     44.5737   
50%        0.0001      0.0001         0.0008      0.0027     52.9376   
75%        0.0011      0.0011         0.0036      0.0036     61.8519   
max        0.1475      0.1475         0.1374      0.0316     96.4039   

         bb_pct_b    hour_sin  
count  11989.0000  11989.0000  
mean       0.5547      0.0006  
std        0.3261      0.7226  
min       -0.5532     -1.0000  
25%        0.3055     -0.707

In [5]:
# ---------- 4. Feature Selection Demo (Strictly on Train Split) ----------
# To prevent lookahead bias (data leakage), feature selection MUST be fitted
# strictly on the training partition, NOT on the whole dataset!
n_total = len(X)
train_size = int(n_total * 0.72)
X_train = X.iloc[:train_size]
y_train = y.iloc[:train_size]

print(f"Running feature selection on training split ({len(X_train):,} samples) ...")
top_features, importances = select_top_features(X_train, y_train, n=20)

print(f"\nTop 15 Most Informative Features (strictly from train set):")
for rank, feat in enumerate(top_features[:15], 1):
    print(f" {rank:2d}. {feat:<22} (Importance: {importances[feat]:.4f})")


Running feature selection on training split (8,632 samples) ...

Top 15 Most Informative Features (strictly from train set):
  1. px_over_ma_20          (Importance: 0.0808)
  2. candle_upper_wick      (Importance: 0.0710)
  3. candle_lower_wick      (Importance: 0.0647)
  4. px_over_ma_50          (Importance: 0.0472)
  5. hl_range               (Importance: 0.0415)
  6. ret_240                (Importance: 0.0391)
  7. ret_lag_1h             (Importance: 0.0359)
  8. ret_13                 (Importance: 0.0357)
  9. macd_norm              (Importance: 0.0323)
 10. bb_width               (Importance: 0.0267)
 11. oc_range               (Importance: 0.0256)
 12. ret_2                  (Importance: 0.0241)
 13. ret_lag_10h            (Importance: 0.0236)
 14. atr_pct_14             (Importance: 0.0218)
 15. macd_hist_norm         (Importance: 0.0212)


In [ ]:
# ---------- 5. Visualize Top Feature Importances ----------
top_imp = importances.head(15).iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(top_imp.index, top_imp.values, color="#3498db", edgecolor="#217dbb")
ax.set_title("Top 15 Feature Importances (Random Forest on Train Set)", fontsize=12, fontweight='bold')
ax.set_xlabel("Relative Importance")
ax.grid(True, alpha=0.3, axis="x")

plt.tight_layout()
feat_plot_path = os.path.join(OUT_DIR, "feature_importances_demo.png")
plt.savefig(feat_plot_path, dpi=200)
plt.show()
print(f"Saved feature importance plot to {feat_plot_path}")
